# Exp 1 - Simulation of Hard, Soft, and Firm Real-Time Tasks using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Model three real-time task classes and evaluate how missed deadlines affect system behaviour.

- **Hard real-time task:** a missed deadline is treated as failure because the system may become unsafe.
- **Firm real-time task:** a late result is discarded because it no longer represents the current state.
- **Soft real-time task:** a late result is still usable, but its quality or value is reduced.

In an autonomous-system context, this distinction matters because the same timing miss can have different consequences depending on whether the task controls braking, planning, logging, display, or telemetry.

## Core Real-Time Systems Theory Notes

### 1. Introduction to Real-Time Systems

A real-time system is a computing system in which correctness depends on two things: the logical correctness of the output and the time at which the output is produced. In a normal general-purpose system, a late answer may be inconvenient. In a real-time system, a late answer may be useless or may cause unsafe behavior.

Logical correctness means that the calculated value or decision is correct. Temporal correctness means that the value or decision is available within the required time bound. A vehicle braking controller, robotic arm controller, industrial motor drive, medical monitoring device, avionics controller, or power-grid protection unit must satisfy both.

Real-time does not simply mean "fast." A fast system that sometimes misses its deadline is not dependable for hard real-time control. A slower system with bounded and predictable timing may be more suitable if it always meets the required deadline. The key engineering properties are determinism, predictability, bounded latency, and analyzable worst-case behavior.

General-purpose systems optimize average response, throughput, fairness, and user convenience. Real-time systems optimize deadline satisfaction, bounded response time, and predictable behavior under defined load. This is why real-time operating systems, embedded controllers, field buses, and deterministic networks often use priority policies, static configuration, time slots, or admission control.

### 2. Classification of Real-Time Systems

Hard real-time systems must not miss deadlines. A missed deadline is treated as system failure. Examples include autonomous emergency braking, airbag control, flight-control surfaces, pacemaker control, and industrial safety shutdown.

Firm real-time systems can tolerate some missed deadlines, but a late result has no value and is discarded. Examples include object-detection frames that arrive after the object is no longer relevant, traffic-sign recognition after the vehicle has passed the sign, or a stale cooperative-awareness message in V2X communication.

Soft real-time systems tolerate deadline misses with quality degradation. Examples include dashboard display refresh, infotainment audio buffering, non-critical telemetry upload, passenger comfort control, and route-estimation updates.

The classification depends on the consequence of lateness, not only the application name. A camera pipeline may be hard real-time when used for emergency braking, firm real-time when used for immediate object tracking, and soft real-time when used for driver display recording.

### 3. Real-Time Tasks and Events

A task is a schedulable unit of computation. In autonomous systems, tasks may represent sensor sampling, frame processing, message transmission, controller update, actuator command generation, logging, or security checking.

Periodic tasks occur at fixed intervals. Example: sample wheel speed every 10 ms. A periodic task is commonly described by execution time C, period T, and deadline D.

Aperiodic tasks occur irregularly and do not have a guaranteed minimum inter-arrival time. Example: a user opens a diagnostic screen. Aperiodic work is often lower criticality or handled by background servers.

Sporadic tasks occur irregularly but have a known minimum separation between arrivals. Example: emergency obstacle events may occur unpredictably but cannot arrive faster than a defined physical or system limit. Sporadic modelling is useful because it allows worst-case analysis.

Time-triggered events are released by a clock schedule. They improve predictability because activation times are known in advance. Event-triggered events are released when an external condition occurs, such as receiving a packet, detecting an obstacle, or crossing a threshold. Event-triggered systems are responsive but require careful overload handling.

### 4. Timing Parameters

The event occurrence time is the real-world time at which the physical event occurs. Release time is when the corresponding task becomes ready for scheduling. Arrival time is often used for the time at which a job enters a queue or a packet reaches a node. Start time is when execution actually begins. Execution time or computation time is the CPU or processor time consumed by the job.

Waiting time is the time spent ready but not executing:

```
waiting_time = start_time - release_time
```

Completion time or finish time is when the job finishes. Response time is the delay from release or arrival to completion:

```
response_time = finish_time - release_time
```

In many lab contexts, turnaround time is also:

```
turnaround_time = finish_time - arrival_time
```

If release time and arrival time are the same, response time and turnaround time become numerically equal. In networked systems they may differ because a real-world event can occur before the software task is released, or a packet can be generated before it reaches the receiving queue.

### 5. Timing Constraints

A relative deadline is measured from release time. An absolute deadline is a time on the system timeline:

```
absolute_deadline = release_time + relative_deadline
```

A deadline is met when:

```
finish_time <= absolute_deadline
```

A deadline miss occurs when:

```
finish_time > absolute_deadline
```

Deadline margin shows how much time remains at completion:

```
deadline_margin = absolute_deadline - finish_time
```

Positive margin means the task finished early. Zero means it finished exactly at the deadline. Negative margin means a miss.

Slack time estimates available spare time before a deadline:

```
slack = absolute_deadline - current_time - remaining_execution_time
```

Laxity is often used similarly:

```
laxity = deadline - current_time - remaining_computation_time
```

Worst-Case Execution Time, or WCET, is the maximum execution time under defined assumptions. Best-Case Execution Time, or BCET, is the minimum. Average execution time is not enough for hard real-time certification because rare long execution paths still matter.

### 6. Communication Performance Parameters

Latency is the time taken for data to move from source to destination:

```
latency = receive_time - send_time
```

Jitter is variation in latency. A simple packet-to-packet jitter estimate is:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Throughput is useful delivered data per unit time:

```
throughput = delivered_bits / observation_time
```

Bandwidth is the nominal or available capacity of a link. Throughput is what is actually achieved after overhead, contention, retransmission, protocol limits, and congestion.

Packet transmission time is:

```
transmission_time = packet_size_bits / link_rate_bits_per_second
```

End-to-end delay can be modeled as:

```
end_to_end_delay = processing_delay + queueing_delay + transmission_delay + propagation_delay
```

Communication overhead is the extra data or time consumed by headers, acknowledgements, encryption, retransmission, routing, and synchronization. Packet loss affects reliability:

```
packet_loss_rate = lost_packets / sent_packets
reliability = delivered_packets / sent_packets
```

### 7. Real-Time Communication Requirements

Bounded latency means there is a known upper limit for message delay under defined conditions. Low jitter means delay stays stable across transmissions. Predictable communication means the designer can reason about message timing before deployment. Reliability means messages are delivered with acceptable probability or with recovery mechanisms. Availability means the communication service is usable when needed.

Deterministic message delivery is often achieved through priority arbitration, time slots, traffic shaping, redundancy, admission control, or real-time Ethernet features. Deadline-aware communication means messages are scheduled according to urgency and usefulness, not simply first-come first-served.

### 8. Timing Analysis in Autonomous Systems

A typical autonomous timing chain is:

```
Sensor -> Perception -> Planning/Control -> Actuator -> Physical Response
```

The perception-to-action delay is:

```
perception_to_action_delay =
    sensor_capture_time
  + sensor_preprocessing_time
  + perception_inference_time
  + planning_time
  + control_time
  + communication_time
  + actuator_response_time
```

For an autonomous braking example:

```
stopping_distance = reaction_distance + braking_distance
reaction_distance = vehicle_speed * total_system_delay
braking_distance = vehicle_speed^2 / (2 * deceleration)
```

Deadline verification compares the computed or measured response time with the maximum safe response time:

```
system_is_timely = measured_response_time <= required_deadline
```

Case Study - Autonomous Emergency Braking:
A front sensor detects an obstacle at a fixed distance. The system must capture sensor data, process it, decide, transmit the command, and apply braking before the remaining stopping distance becomes unsafe. The case study shows why real-time correctness is a chain property. A fast perception algorithm alone is not enough if the actuator command is delayed.

Case Study - Robotic Arm in Industrial Automation:
A robotic arm must stop when a worker crosses a safety boundary. Sensor detection, controller scheduling, network delivery, and motor-drive response must all be bounded. High average throughput is irrelevant if one delayed safety packet allows the arm to continue moving too long.

Case Study - V2X Hazard Warning:
A vehicle broadcasts a hazard message to nearby vehicles. The message is useful only if received before the receiving vehicle must react. This connects communication latency, jitter, packet loss, message freshness, and security verification.

### 9. Textbook Design Workflow for Real-Time Experiments

When solving a real-time lab problem, use a disciplined workflow. First identify the physical event or communication event. Second identify the software task or network message created by that event. Third list the timing parameters: release time, start time, execution time, finish time, and deadline. Fourth compute the response time and deadline margin. Fifth classify the consequence of lateness as hard, firm, or soft. Sixth propose a design improvement if the deadline is missed.

For autonomous systems, the timing boundary should be tied to a physical reason. For example, a braking deadline should relate to speed, distance, and deceleration. A communication deadline should relate to how long a message remains useful. A security verification deadline should relate to whether authentication or IDS checks finish before the receiver uses the message.

### 10. Common Architectures Used Across These Experiments

Most experiments in this lab can be understood using one of three architecture patterns.

Control-loop pattern:

```
Sensor -> Controller Task -> Actuator -> Plant / Vehicle -> Sensor
```

Communication-loop pattern:

```
Publisher / Sender -> Network Medium -> Receiver / Subscriber -> Application Decision
```

Security-monitoring pattern:

```
Message Source -> Security Check -> IDS / Risk Logic -> Accept, Reject, or Alert
```

The control-loop pattern focuses on WCET, response time, and deadline satisfaction. The communication-loop pattern focuses on latency, jitter, throughput, packet loss, and deterministic delivery. The security-monitoring pattern focuses on integrity, authentication, replay resistance, anomaly detection, and risk reduction. Autonomous systems usually combine all three patterns, which is why timing and security cannot be treated as separate afterthoughts.

### 11. Common Mistakes to Avoid in Lab Answers

Do not say "real-time means fast." Say "real-time means deadline-bound." Do not use average execution time as a substitute for WCET in hard real-time analysis. Do not conclude that high throughput guarantees good real-time performance. Do not claim a security mechanism provides authentication unless the mechanism actually proves sender identity. Do not claim a physical simulator or broker was used if the notebook uses a Python fallback. Clear assumptions make the lab record more credible.

### Core References for These Notes

- Python timing functions such as `perf_counter()` and monotonic clocks are documented by the official Python `time` module documentation: https://docs.python.org/3/library/time.html
- IEEE 802.1 Time-Sensitive Networking is the IEEE working-group area for time-sensitive network behavior: https://1.ieee802.org/tsn/
- SUMO official documentation describes traffic simulation concepts used in V2V mobility experiments: https://sumo.dlr.de/docs/
- MQTT is an OASIS publish-subscribe messaging standard for IoT telemetry: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- NIST FIPS 180-4 specifies SHA-256 as part of the Secure Hash Standard: https://csrc.nist.gov/pubs/fips/180-4/upd1/final
- NIST SP 800-30 Rev. 1 provides risk-assessment guidance: https://csrc.nist.gov/pubs/sp/800/30/r1/final

## Extended Experiment Notes and Case Studies

### Experiment Focus

This experiment demonstrates hard, firm, and soft real-time task behavior using Python simulation. The goal is not to prove that Python is a hard real-time platform. Python is used here as an educational simulator to calculate deadlines, completion times, misses, and usefulness of late results.

### Experiment Architecture

```
Task Definition Table
  -> Task Release Timeline
  -> Scheduler / Execution Simulator
  -> Finish-Time Calculator
  -> Deadline Checker
  -> Result Classifier: hard failure, firm discard, soft degradation
```

The architecture separates task properties from timing analysis. Each task has a release time, execution time, deadline, and class. The simulator computes the finish time and checks whether the result is valid for that task class.

### Detailed Formula Set

```
absolute_deadline = release_time + relative_deadline
finish_time = start_time + execution_time
response_time = finish_time - release_time
deadline_margin = absolute_deadline - finish_time
miss_ratio = missed_tasks / total_tasks
useful_result_ratio = accepted_results / total_results
```

For a hard task, any negative deadline margin must be treated as unacceptable. For a firm task, a negative margin means the output is discarded. For a soft task, a negative margin means the output may still be used, but quality is reduced.

### Case Study 1 - Emergency Braking Controller

A braking command has a strict deadline derived from vehicle speed, obstacle distance, and braking capability. If computation finishes after the deadline, the command may not prevent collision. This is hard real-time. The lab record should explain that the failure comes from lateness, even if the brake decision itself is logically correct.

### Case Study 2 - Camera Frame for Object Tracking

An object-detection frame may be useful only until the next frame or until the object position changes significantly. If it arrives late, it is discarded. This is firm real-time. The key metric is not only how many frames were processed, but how many were processed before their usefulness expired.

### Case Study 3 - Infotainment Display

A map refresh or passenger display can miss a frame deadline without creating immediate danger. This is soft real-time. The observed effect is reduced smoothness, lag, or user discomfort.

### Lab Record Guidance

Write a table with task name, task class, release time, execution time, relative deadline, absolute deadline, finish time, margin, and final status. The conclusion must compare the three classes and explain why the same amount of lateness has different consequences.

## Detailed Notes

### 1. Real-Time Task Classes

Real-time systems are judged not only by whether a computation is correct, but also by whether the answer is produced before its deadline.

| Task Class | Meaning | Deadline Miss Effect | Autonomous-System Example |
|---|---|---|---|
| Hard real-time | Deadline must always be met | Unsafe or failed system behaviour | Emergency braking, steering actuation |
| Firm real-time | Late output has no value | Result is discarded | Stale obstacle-detection frame |
| Soft real-time | Late output still has partial value | Quality degrades | Telemetry display, video stream |

### 2. Architecture Explanation

The experiment models a small real-time execution pipeline:

```text
Task Definition
  -> Scheduler
  -> Execution Timeline
  -> Deadline Checker
  -> Class-wise Result Analysis
```

Each task contains a release time, execution time, relative deadline, and task class. The scheduler simulates when the task starts and finishes. The deadline checker decides whether the output is on time. The final analysis explains the result differently for hard, firm, and soft tasks.

### 3. Key Timing Terms

- **Release time (R):** time at which the task becomes ready.
- **Execution time (C):** processor time required by the task.
- **Relative deadline (Drel):** allowed time after release.
- **Absolute deadline (Dabs):** final completion time allowed on the global timeline.
- **Finish time (F):** time at which the task actually completes.

### 4. Formulas

Absolute deadline:

\[
D_{abs} = R + D_{rel}
\]

Deadline condition:

\[
F \le D_{abs}
\]

Deadline miss ratio:

\[
\text{miss ratio} = \frac{\text{number of missed tasks}}{\text{total number of tasks}}
\]

Deadline satisfaction ratio:

\[
\text{satisfaction ratio} = \frac{\text{number of tasks completed before deadline}}{\text{total number of tasks}}
\]

Useful-work score used in the post-lab:

\[
\text{useful work ratio} = \frac{\sum \text{useful credit}}{\text{number of tasks}}
\]

The notebook gives full credit to on-time results, no credit to missed hard/firm tasks, and partial credit to missed soft tasks.

### 5. In-Lab Interpretation

The in-lab simulation uses a simple non-preemptive scheduler. This means tasks execute one after another. If early jobs consume too much time, later jobs can miss their deadlines.

Important observation:

- Hard tasks should be designed so their deadlines are met even under load.
- Firm tasks can complete late, but their results are useless after the deadline.
- Soft tasks can complete late and still provide partial value.

### 6. Post-Lab Interpretation

The post-lab experiment increases system load from 40% to 120%. Increasing load increases actual execution time. When the system becomes overloaded, the miss ratio rises.

What to observe:

- At low load, all task classes usually meet deadlines.
- Near full load, deadline misses begin appearing.
- In overload, firm and soft tasks lose useful output.
- A hard-task miss is unacceptable even if the numerical miss ratio is small.

### 7. Practical Design Notes

For an autonomous system, the scheduler should not treat all tasks equally. Safety-critical tasks need stronger guarantees through priority assignment, admission control, execution-time budgeting, or dedicated compute resources. Non-critical telemetry or display tasks can be degraded first during overload.

### 8. Precautions

- Do not interpret this simple scheduler as a production RTOS scheduler.
- Use worst-case execution time, not average execution time, for safety-critical design.
- Validate deadline assumptions with real measurements before using them in a deployed system.
- Keep hard real-time workloads small and predictable.


## Architecture

```text
Task Set
  |-- task name
  |-- class: hard / firm / soft
  |-- release time
  |-- execution time
  |-- relative deadline
          |
          v
Simple Scheduler
  |-- waits until task release
  |-- executes task for its execution time
  |-- records finish time
          |
          v
Deadline Evaluator
  |-- absolute deadline = release time + relative deadline
  |-- deadline met if finish time <= absolute deadline
          |
          v
Result Table + Class-wise Miss Ratio
```

The notebook uses a non-preemptive scheduler so the timing effect is easy to inspect. In a real system, higher-priority hard real-time tasks would normally receive stricter scheduling guarantees.

## Formulas and Required Theory

For a task \(i\):

\[
D_i^{abs} = R_i + D_i^{rel}
\]

\[
\text{deadline met}_i =
\begin{cases}
1, & F_i \le D_i^{abs}\\
0, & F_i > D_i^{abs}
\end{cases}
\]

\[
\text{deadline miss ratio} = \frac{\text{number of missed jobs}}{\text{total number of jobs}}
\]

\[
\text{useful work ratio} =
\frac{\sum \text{useful credit per completed job}}{\text{total jobs}}
\]

Useful-credit rule used in this notebook:

- on-time result: `1.00`
- late hard result: `0.00`
- late firm result: `0.00`
- late soft result: `0.55`, representing degraded but still partially useful output

## In-Lab Method

1. Define task parameters for representative autonomous-system workloads.
2. Execute the task list in scheduler order.
3. Compute finish time for each task.
4. Compare finish time against absolute deadline.
5. Report per-task status and class-wise miss ratio.

Expected observation: hard tasks should be protected from misses. Firm and soft tasks may miss under overload, but the interpretation differs: firm results are discarded while soft results degrade.

In [1]:
tasks = [
    {"name": "Brake actuator command", "class": "Hard", "release": 0, "exec": 6, "deadline": 8},
    {"name": "Collision-warning fusion", "class": "Hard", "release": 2, "exec": 5, "deadline": 12},
    {"name": "Trajectory replanning", "class": "Firm", "release": 4, "exec": 9, "deadline": 14},
    {"name": "HD map refresh", "class": "Firm", "release": 7, "exec": 12, "deadline": 18},
    {"name": "Cabin telemetry update", "class": "Soft", "release": 8, "exec": 15, "deadline": 20},
    {"name": "Video status stream", "class": "Soft", "release": 10, "exec": 18, "deadline": 22},
]

clock = 0
results = []
for task in tasks:
    start = max(clock, task["release"])
    finish = start + task["exec"]
    absolute_deadline = task["release"] + task["deadline"]
    met = finish <= absolute_deadline
    results.append((task["name"], task["class"], task["release"], task["exec"], absolute_deadline, finish, "Met" if met else "Missed"))
    clock = finish

print("EXP 1 - IN-LAB RESULT")
print(f"{'Task':32} {'Class':6} {'Rel':>4} {'Exec':>5} {'Dead':>5} {'Fin':>5} {'Status':>8}")
for row in results:
    print(f"{row[0][:32]:32} {row[1]:6} {row[2]:4} {row[3]:5} {row[4]:5} {row[5]:5} {row[6]:>8}")
for cls in ["Hard", "Firm", "Soft"]:
    subset = [r for r in results if r[1] == cls]
    misses = sum(r[-1] == "Missed" for r in subset)
    print(f"{cls} miss ratio = {misses / len(subset):.2f}")

EXP 1 - IN-LAB RESULT
Task                             Class   Rel  Exec  Dead   Fin   Status
Brake actuator command           Hard      0     6     8     6      Met
Collision-warning fusion         Hard      2     5    14    11      Met
Trajectory replanning            Firm      4     9    18    20   Missed
HD map refresh                   Firm      7    12    25    32   Missed
Cabin telemetry update           Soft      8    15    28    47   Missed
Video status stream              Soft     10    18    32    65   Missed
Hard miss ratio = 0.00
Firm miss ratio = 1.00
Soft miss ratio = 1.00


## Post-Lab Method

The post-lab cell generates 100 mixed tasks at several load levels. Load scales actual execution time. This shows how increasing computational demand affects deadline misses and useful work.

Interpretation rule:

- A rising miss ratio means the system is approaching overload.
- A hard-task miss ratio above zero is unacceptable in a safety-critical design.
- A firm-task miss reduces useful output sharply.
- A soft-task miss reduces quality gradually.

In [2]:
import random

def run_trial(load, seed=341401):
    rng = random.Random(seed)
    factors = {"Hard": 1.75, "Firm": 1.35, "Soft": 1.25}
    counts = {k: 0 for k in factors}
    misses = {k: 0 for k in factors}
    useful = {k: 0.0 for k in factors}
    for i in range(100):
        cls = ["Hard", "Firm", "Soft"][i % 3]
        base = rng.uniform(4, 18)
        actual = base * load * rng.uniform(0.90, 1.35)
        met = actual <= base * factors[cls]
        counts[cls] += 1
        misses[cls] += 0 if met else 1
        useful[cls] += 1.0 if met else (0.55 if cls == "Soft" else 0.0)
    return {cls: (misses[cls] / counts[cls], useful[cls] / counts[cls]) for cls in counts}

print("EXP 1 - POST-LAB LOAD SWEEP")
print(f"{'Load':>6} {'Class':>6} {'Miss ratio':>12} {'Useful work':>12} {'Plot':>10}")
for load in [0.40, 0.60, 0.80, 1.00, 1.20]:
    for cls, values in run_trial(load).items():
        miss, useful = values
        plot = "-" if miss == 0 else "#" * max(1, round(miss * 10))
        print(f"{load:6.0%} {cls:>6} {miss:12.2f} {useful:12.2f} {plot:>10}")

EXP 1 - POST-LAB LOAD SWEEP
  Load  Class   Miss ratio  Useful work       Plot
   40%   Hard         0.00         1.00          -
   40%   Firm         0.00         1.00          -
   40%   Soft         0.00         1.00          -
   60%   Hard         0.00         1.00          -
   60%   Firm         0.00         1.00          -
   60%   Soft         0.00         1.00          -
   80%   Hard         0.00         1.00          -
   80%   Firm         0.00         1.00          -
   80%   Soft         0.00         1.00          -
  100%   Hard         0.00         1.00          -
  100%   Firm         0.00         1.00          -
  100%   Soft         0.09         0.96          #
  120%   Hard         0.00         1.00          -
  120%   Firm         0.61         0.39     ######
  120%   Soft         0.61         0.73     ######


## What to Write in the Lab Record

- Copy the in-lab result table.
- Record which classes met or missed deadlines.
- Explain why hard tasks need the strongest timing guarantee.
- In the post-lab section, compare miss ratio and useful work at 40%, 60%, 80%, 100%, and 120% load.
- Conclude that deadline classification changes the engineering response to overload.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html